# Task 1: Medical Image Classification (Chest X-Ray: Pneumonia vs Normal)
**Goal:** Classify chest X-ray images as **NORMAL** or **PNEUMONIA**.

**Dataset:** Chest X-Ray Images (Pneumonia), Kaggle (about 5,800 images).

**Approach:** Transfer learning with **MobileNetV2** (pre-trained on ImageNet), data augmentation, class weights for the imbalanced classes, and a short fine-tuning phase. Evaluated with accuracy, precision, recall, F1, confusion matrix and ROC-AUC (in medical problems, **recall for the disease class** is especially important).

**Before running in Colab:** Runtime, Change runtime type, choose **T4 GPU**. Put your dataset zip in Google Drive.

## 1. Get the dataset into Colab
Upload the zip to Google Drive first (much faster than uploading a 2 GB file into Colab directly), then change `ZIP_PATH` to your file's name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/archive.zip"   # <-- change to your zip file name
!cp "{ZIP_PATH}" /content/data.zip
!unzip -q -o /content/data.zip -d /content/data
print("Done")

## 2. Import libraries

In [ ]:
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = (224, 224)
BATCH = 32
print("TensorFlow:", tf.__version__)
print("GPU found:", tf.config.list_physical_devices("GPU"))

## 3. Find the data folders and count the images

In [ ]:
DATA_DIR = "/content/data"

def find_root(base):
    for dirpath, dirnames, _ in os.walk(base):
        if "__MACOSX" in dirpath:
            continue
        if "train" in dirnames and "test" in dirnames:
            return dirpath
    raise FileNotFoundError("Could not find train/ and test/ folders. Check the unzip step.")

root = find_root(DATA_DIR)
class_names = sorted(d for d in os.listdir(os.path.join(root, "train"))
                     if os.path.isdir(os.path.join(root, "train", d)))
print("Dataset folder:", root)
print("Classes:", class_names)

EXTS = (".jpg", ".jpeg", ".png")
def list_images(split):
    rows = []
    for idx, cls in enumerate(class_names):
        for f in glob.glob(os.path.join(root, split, cls, "*")):
            if f.lower().endswith(EXTS):
                rows.append((f, idx))
    return pd.DataFrame(rows, columns=["path", "label"])

all_train = list_images("train")
test_df = list_images("test")
print("\nTrain images:", len(all_train), "| Test images:", len(test_df))
print(all_train.label.map(dict(enumerate(class_names))).value_counts())

## 4. Look at some images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.ravel(), all_train.sample(8, random_state=SEED).iterrows()):
    ax.imshow(plt.imread(row.path), cmap="gray")
    ax.set_title(class_names[row.label]); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train / validation / test split
The official `val` folder has only 16 images, which is too small to be reliable. So **15% of the training data is used for validation**, and the official `test` folder is used only for the final evaluation.

In [ ]:
train_df, val_df = train_test_split(all_train, test_size=0.15,
                                    stratify=all_train.label, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

## 6. Build the input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label

def make_ds(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, df.label.values.astype("float32")))
    if training:
        ds = ds.shuffle(len(df), seed=SEED)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_df, training=True)
val_ds = make_ds(val_df)
test_ds = make_ds(test_df)

## 7. Build the model (MobileNetV2 transfer learning)

In [ ]:
def build_model():
    base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,),
                                             include_top=False, weights="imagenet")
    base.trainable = False                      # freeze the pre-trained layers first

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = layers.RandomFlip("horizontal")(inputs)  # augmentation (only active during training)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.Rescaling(1./127.5, offset=-1)(x) # MobileNetV2 expects pixels in [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inputs, outputs), base

model, base = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
model.summary()

## 8. Train: phase 1 (only the new top layers)
The dataset has more PNEUMONIA than NORMAL images, so **class weights** are used to balance them.

In [ ]:
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df.label.values)
class_weight = {0: cw[0], 1: cw[1]}
print("Class weights:", class_weight)

es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=5,
               class_weight=class_weight, callbacks=[es])

## 9. Train: phase 2 (fine-tune the last layers of MobileNetV2 with a small learning rate)

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=3,
               class_weight=class_weight, callbacks=[es])

## 10. Training curves

In [ ]:
def join(key):
    return h1.history[key] + h2.history[key]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(join("accuracy"), label="train"); ax[0].plot(join("val_accuracy"), label="validation")
ax[0].axvline(len(h1.history["accuracy"]) - 1, color="gray", linestyle="--", label="fine-tuning starts")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(join("loss"), label="train"); ax[1].plot(join("val_loss"), label="validation")
ax[1].axvline(len(h1.history["loss"]) - 1, color="gray", linestyle="--")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.show()

## 11. Evaluate on the test set

In [ ]:
y_prob = model.predict(test_ds).ravel()
y_true = test_df.label.values
y_pred = (y_prob >= 0.5).astype(int)

test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test ROC-AUC: {test_auc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix (test set)")
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curve"); plt.legend(); plt.show()

## 12. Sample predictions

In [ ]:
sample = test_df.sample(8, random_state=1)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = tf.image.resize(tf.io.decode_image(tf.io.read_file(row.path), channels=3, expand_animations=False), IMG_SIZE)
    p = float(model.predict(tf.expand_dims(img, 0), verbose=0)[0, 0])
    pred = class_names[int(p >= 0.5)]
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(f"Actual: {class_names[row.label]}\nPredicted: {pred} ({p:.2f})",
                 color="green" if pred == class_names[row.label] else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 13. Results and Conclusion
*(Fill in the blanks from the outputs of section 11 after running the notebook.)*

| Metric | Value |
|--------|-------|
| Test accuracy | ____ |
| ROC-AUC | ____ |
| PNEUMONIA recall | ____ |
| PNEUMONIA precision | ____ |
| NORMAL recall | ____ |

**Observations:**
- Transfer learning with MobileNetV2 reached ____% accuracy on the unseen test set.
- The model catches ____% of the pneumonia cases (recall). Missing a real case is the most costly mistake in a medical setting.
- The most common error was ____ (see the confusion matrix).

**Conclusion:** A pre-trained MobileNetV2 with augmentation, class weights and fine-tuning can classify chest X-rays effectively. This is an educational project and **not a medical diagnosis tool**.

**Future work:** Try larger models (ResNet50, EfficientNet), use Grad-CAM to see which regions the model looks at, and validate on data from other hospitals.